# RAG Evaluation using DeepEval Metrics

This notebook demonstrates how to evaluate a RAG (Retrieval-Augmented Generation) pipeline using the [DeepEval](https://docs.confident-ai.com/) framework with the **NESGEN** (Nestlé GenAI) LLM as the evaluation model.

DeepEval is an open-source LLM evaluation framework offering:
- **14+ built-in metrics** covering quality, relevancy, retrieval, safety, and more
- **LLM-as-a-judge** evaluation with deterministic reasoning
- **Pytest integration** for CI/CD pipelines
- **Synthetic dataset generation** capabilities

The scoring information can optionally be stored in [Langfuse](https://langfuse.com) for further analysis.

**Notes**
- DeepEval metrics use LLM-as-a-judge and incur API calls to NESGEN
- Start with a small dataset (1–3 items) to validate your setup before scaling
- Metrics that require `expected_output` (e.g., `ContextualRecall`) need ground-truth answers

## Prerequisites

Before using this notebook:
- Understand the [DeepEval metrics overview](https://docs.confident-ai.com/docs/metrics-introduction)
- Have your NESGEN `client_id` and `client_secret` available
- (Optional) Have Langfuse credentials if you want to log results to a dashboard
- Perform the pre-requisite steps in the [README](../README.md)

## 1. Installation

Install required packages.

In [ ]:
!uv pip install -q langchain-openai langfuse ragas deepeval

## 2. Environment Setup

Load credentials from a `.env` file or enter them interactively.

**Recommended:** Copy `.env.example` from the repo root, rename to `.env`, and fill in the values.

In [ ]:
from __future__ import annotations

import getpass
import os
import sys

from dotenv import load_dotenv

# Load environment variables from .env file (if present)
load_dotenv()


def get_env_or_prompt(var_name: str, prompt_text: str, is_secret: bool = True) -> str:
    """Return env var value or interactively prompt the user."""
    value = os.getenv(var_name)
    if value:
        return value
    if is_secret:
        return getpass.getpass(prompt_text)
    return input(prompt_text)


# ── NESGEN credentials ──────────────────────────────────────────────────────
os.environ["NESGEN_CLIENT_ID"] = get_env_or_prompt(
    "NESGEN_CLIENT_ID", "Enter NESGEN Client ID: "
)
os.environ["NESGEN_CLIENT_SECRET"] = get_env_or_prompt(
    "NESGEN_CLIENT_SECRET", "Enter NESGEN Client Secret: "
)

# ── Langfuse credentials (optional) ────────────────────────────────────────
langfuse_public_key = os.getenv("LANGFUSE_PUBLIC_KEY", "")
langfuse_secret_key = os.getenv("LANGFUSE_SECRET_KEY", "")
langfuse_host       = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")

USE_LANGFUSE = bool(langfuse_public_key and langfuse_secret_key)
print(f"Langfuse integration: {'enabled' if USE_LANGFUSE else 'disabled (no credentials found)'}")
print("NESGEN credentials loaded.")

## 3. Add Project Root to Path

The NESGEN wrappers and config live under `metric_evaluation/`. We add that to `sys.path` so the imports resolve correctly.

In [ ]:
# Resolve path to metric_evaluation so we can import src modules
METRIC_EVAL_DIR = os.path.abspath(os.path.join("..", "..", "metric_evaluation"))
if METRIC_EVAL_DIR not in sys.path:
    sys.path.insert(0, METRIC_EVAL_DIR)

print(f"Added to sys.path: {METRIC_EVAL_DIR}")

## 4. The Dataset

For this example we use the [FIQA (Financial Opinion Mining and Question Answering)](https://huggingface.co/datasets/explodinggradients/fiqa) dataset from Hugging Face.

The dataset contains:
- `question`: The user question
- `answer`: The answer generated by the RAG pipeline
- `contexts`: The document chunks retrieved for that question
- `ground_truths`: The reference answer(s)

See the [Hugging Face dataset page](https://huggingface.co/datasets/explodinggradients/fiqa) for full details.

In [ ]:
from typing import Any, Dict

from datasets import load_dataset

# Load FIQA dataset — same source as the RAGAS cookbook
# Limit to 3 items for testing; remove .select() for a full run
DEMO_LIMIT = 3
fiqa_eval = load_dataset("explodinggradients/fiqa", "ragas_eval")["baseline"].select(
    range(DEMO_LIMIT)
)

print(f"Loaded {len(fiqa_eval)} items from FIQA dataset")
print("\nSample item:")
print(f"  Q: {fiqa_eval[0]['question']}")
print(f"  A: {fiqa_eval[0]['answer'][:80]}...")
print(f"  Contexts: {len(fiqa_eval[0]['contexts'])} retrieved chunks")
print(f"  Ground truths: {fiqa_eval[0]['ground_truths']}")

In [ ]:
# Convert FIQA dataset to the eval_dataset format expected by DeepEval.
# FIQA already contains pre-computed answers and contexts from a RAG pipeline,
# so no local retriever is needed.
eval_dataset = []
for item in fiqa_eval:
    item_dict: Dict[str, Any] = dict(item)
    eval_dataset.append(
        {
            "question":     item_dict["question"],
            "answer":       item_dict["answer"],
            "contexts":     item_dict["contexts"],
            # ground_truths is a list in FIQA; take the first entry
            "ground_truth": item_dict["ground_truths"][0] if item_dict["ground_truths"] else "",
        }
    )

print(f"Prepared {len(eval_dataset)} evaluation items.")
for i, item in enumerate(eval_dataset):
    label = item["question"][:80] + "..." if len(item["question"]) > 80 else item["question"]
    print(f"\nItem {i + 1}: {label}")
    print(f"  Retrieved {len(item['contexts'])} context chunks")

## 5. NESGEN Model Wrapper for DeepEval

DeepEval uses an LLM judge to score test cases. We plug in the **NESGEN** model via the `NESGENDeepEvalModel` wrapper (defined in `metric_evaluation/src/deepeval_metrics.py`), which extends `DeepEvalBaseLLM`.

In [ ]:
from src.deepeval_metrics import NESGENDeepEvalModel

# Initialise the NESGEN judge model
nesgen_judge = NESGENDeepEvalModel()
print(f"Judge model: {nesgen_judge.get_model_name()}")

# Quick sanity check
test_response = nesgen_judge.generate("Respond with exactly: hello")
print(f"Sanity check response: {test_response[:100]}")

## 6. Define DeepEval Metrics

We configure the following metrics, all powered by the NESGEN judge:

| Metric | What it measures | Requires |
|--------|-----------------|----------|
| **Faithfulness** | Whether the answer is grounded in the retrieved contexts | `input`, `actual_output`, `retrieval_context` |
| **Answer Relevancy** | How relevant the answer is to the question | `input`, `actual_output` |
| **Contextual Precision** | Are the most relevant chunks ranked highest? | `input`, `actual_output`, `retrieval_context`, `expected_output` |
| **Contextual Recall** | Does the retrieved context cover the ground truth? | `retrieval_context`, `expected_output` |
| **Hallucination** | Does the answer contain facts not in the context? | `actual_output`, `context` |

> **Note:** `BiasMetric` and `ToxicityMetric` are excluded. Their built-in evaluation templates contain politically sensitive example text that triggers NESGEN's content policy filter, returning a `400 Bad Request` error. These metrics can be re-enabled if the NESGEN policy is updated or a custom template is used.

In [ ]:
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    HallucinationMetric,
)

THRESHOLD = 0.7  # Minimum acceptable score for all metrics

# Instantiate metrics — each uses NESGEN as the judge LLM
# Note: BiasMetric and ToxicityMetric are excluded because their built-in templates
# contain sensitive example text that triggers NESGEN's content policy (400 Bad Request).
faithfulness_metric         = FaithfulnessMetric(threshold=THRESHOLD,  model=nesgen_judge)
answer_relevancy_metric     = AnswerRelevancyMetric(threshold=THRESHOLD, model=nesgen_judge)
contextual_precision_metric = ContextualPrecisionMetric(threshold=THRESHOLD, model=nesgen_judge)
contextual_recall_metric    = ContextualRecallMetric(threshold=THRESHOLD, model=nesgen_judge)
hallucination_metric        = HallucinationMetric(threshold=THRESHOLD,  model=nesgen_judge)

# Group metrics for convenience
ALL_METRICS = [
    faithfulness_metric,
    answer_relevancy_metric,
    contextual_precision_metric,
    contextual_recall_metric,
    hallucination_metric,
]

print(f"Configured {len(ALL_METRICS)} metrics with threshold={THRESHOLD}")

## 7. Create DeepEval Test Cases

`LLMTestCase` is DeepEval's unit of evaluation. Each test case contains:
- `input` — the user question
- `actual_output` — the RAG-generated answer
- `retrieval_context` — the retrieved document chunks
- `expected_output` — the ground-truth answer (optional but required by some metrics)

In [ ]:
from deepeval.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input=item["question"],
        actual_output=item["answer"],
        retrieval_context=item["contexts"],
        expected_output=item.get("ground_truth", ""),
        # context is required by HallucinationMetric (same as retrieval_context)
        context=item["contexts"],
    )
    for item in eval_dataset
]

print(f"Created {len(test_cases)} test cases.")
print("\nFirst test case:")
tc = test_cases[0]
print(f"  Input:           {tc.input[:80]}")
print(f"  Actual output:   {tc.actual_output[:80]}")
print(f"  Expected output: {tc.expected_output[:80]}")
print(f"  # Contexts:      {len(tc.retrieval_context)}")

## 8. Single Test Case Evaluation

Evaluate the first test case against all metrics to verify the setup.



#Might run into some issues while running the notebook 

Option 1: Disable DeepEval telemetry (recommended)

Add this at the top of the notebook before importing deepeval:


import os
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"


Option 2: Add the corporate CA certificate to Python's trust store


import os
os.environ["REQUESTS_CA_BUNDLE"] = r"C:\path\to\corporate-ca-bundle.crt"
os.environ["SSL_CERT_FILE"] = r"C:\path\to\corporate-ca-bundle.crt"
The .crt file can usually be exported from the Windows Certificate Store (certmgr.msc → Trusted Root Certification Authorities).

In [ ]:
from typing import Any


def evaluate_test_case(test_case: LLMTestCase, metrics: list) -> dict[str, Any]:
    """Run all metrics against a single test case and return a results dict."""
    results = {}
    for metric in metrics:
        metric_name = metric.__class__.__name__.replace("Metric", "").lower()
        try:
            metric.measure(test_case)
            results[metric_name] = {
                "score":   round(metric.score, 4),
                "passed":  metric.is_successful(),
                "reason":  metric.reason,
            }
        except Exception as exc:
            results[metric_name] = {
                "score":  0.0,
                "passed": False,
                "reason": f"Error: {exc}",
            }
    return results


print("Evaluating first test case...\n")
single_results = evaluate_test_case(test_cases[0], ALL_METRICS)

for metric_name, data in single_results.items():
    status = "PASS" if data["passed"] else "FAIL"
    print(f"  [{status}] {metric_name:<25} score={data['score']:.4f}")
    if data["reason"]:
        print(f"           reason: {str(data['reason'])[:120]}")

## 9. Batch Evaluation

Evaluate all test cases and collect per-item scores.

In [ ]:
import pandas as pd

rows = []
for i, test_case in enumerate(test_cases):
    print(f"Evaluating test case {i+1}/{len(test_cases)}: {test_case.input[:60]}...")
    results = evaluate_test_case(test_case, ALL_METRICS)
    row = {
        "question": test_case.input,
        "answer":   test_case.actual_output[:80] + "..." if len(test_case.actual_output) > 80 else test_case.actual_output,
    }
    for metric_name, data in results.items():
        row[f"{metric_name}_score"]  = data["score"]
        row[f"{metric_name}_passed"] = data["passed"]
    rows.append(row)

results_df = pd.DataFrame(rows)
print("\nEvaluation complete!")
results_df

## 10. Aggregate Results & Summary

Compute average scores across all test cases and compare against the threshold.

In [ ]:
if "results_df" not in dir():
    raise RuntimeError(
        "results_df is not defined — run the batch evaluation cell (Section 9) first."
    )

score_cols = [c for c in results_df.columns if c.endswith("_score")]
avg_scores = results_df[score_cols].mean().rename(lambda c: c.replace("_score", ""))

print("=" * 60)
print("DEEPEVAL RAG EVALUATION — AVERAGE SCORES")
print("=" * 60)
print(f"{'Metric':<30} {'Avg Score':>10}  {'Status':>8}")
print("-" * 60)
for metric, avg in avg_scores.items():
    status = "PASS" if avg >= THRESHOLD else "FAIL"
    print(f"{metric:<30} {avg:>10.4f}  {status:>8}")
print("=" * 60)

overall_pass = all(v >= THRESHOLD for v in avg_scores)
print(f"\nOverall evaluation: {'PASSED' if overall_pass else 'FAILED'} (threshold={THRESHOLD})")

In [ ]:
#!pip install matplotlib

In [ ]:
# Visualise scores as a bar chart (requires matplotlib)
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ["green" if v >= THRESHOLD else "red" for v in avg_scores.values]
    bars = ax.bar(avg_scores.index, avg_scores.values, color=colors, edgecolor="black", linewidth=0.5)
    ax.axhline(y=THRESHOLD, color="orange", linestyle="--", linewidth=1.5, label=f"Threshold ({THRESHOLD})")
    ax.set_ylim(0, 1.05)
    ax.set_title("DeepEval RAG Metric Scores (NESGEN Judge)", fontsize=14, fontweight="bold")
    ax.set_xlabel("Metric")
    ax.set_ylabel("Average Score")
    ax.legend()
    plt.xticks(rotation=30, ha="right")
    for bar, val in zip(bars, avg_scores.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — skipping chart. Run: pip install matplotlib")

## 11. Using the DeepEvalEvaluator Class (High-level API)

The project ships a higher-level `DeepEvalEvaluator` class in `src/deepeval_metrics.py` that wraps the above steps and provides Langfuse integration automatically.

In [ ]:
from src.deepeval_metrics import DeepEvalEvaluator

evaluator = DeepEvalEvaluator(threshold=THRESHOLD)
print(f"Langfuse: {'connected' if evaluator.langfuse else 'file fallback (no credentials)'}")

In [ ]:
# Evaluate with DataFrame output (detailed per-item results)
detailed_df = evaluator.evaluate_with_dataframe(
    eval_data=eval_dataset,
    metrics=["faithfulness", "answer_relevancy", "hallucination"],  # subset for speed
)

print("Per-item detailed results:")
detailed_df

In [ ]:
# Batch evaluation with aggregated scores
questions  = [item["question"]     for item in eval_dataset]
answers    = [item["answer"]       for item in eval_dataset]
contexts   = [item["contexts"]     for item in eval_dataset]
gt_answers = [item["ground_truth"] for item in eval_dataset]

aggregated = evaluator.evaluate_batch(
    questions=questions,
    answers=answers,
    contexts=contexts,
    ground_truths=gt_answers,
    metrics=["faithfulness", "answer_relevancy", "hallucination"],
)

# Pretty-print summary
print(evaluator.get_metrics_summary(aggregated))

In [ ]:
# Check which metrics passed the threshold
passed = evaluator.compare_with_threshold(aggregated)
print("Threshold check results:")
for metric, ok in passed.items():
    print(f"  {metric:<30} {'PASS' if ok else 'FAIL'}")

## 12. Optional: Run via DeepEval's Native `evaluate()` API

DeepEval also has a built-in `evaluate()` function that runs all metrics in parallel and produces a structured report. This is the recommended approach for CI/CD integration.

In [ ]:
from deepeval import evaluate
from deepeval.evaluate.configs import DisplayConfig

# Run native DeepEval evaluate — generates a full HTML/JSON test report
# Note: uses ALL_METRICS defined earlier
# In deepeval 3.x, print_results moved into DisplayConfig
eval_results = evaluate(
    test_cases=test_cases,
    metrics=ALL_METRICS,
    display_config=DisplayConfig(print_results=True),
)
print("\nNative evaluation complete.")

## 13. Optional: Log Results to Langfuse(Yet to be tested once the Containerized instance is ready)

If Langfuse credentials are configured, we push the evaluation scores to a Langfuse trace for dashboarding.

In [ ]:
if USE_LANGFUSE:
    from langfuse import Langfuse

    lf = Langfuse(
        public_key=langfuse_public_key,
        secret_key=langfuse_secret_key,
        host=langfuse_host,
    )

    if lf.auth_check():
        print("Langfuse authenticated — logging evaluation scores...")

        trace = lf.trace(
            name="deepeval_rag_evaluation",
            metadata={
                "framework":    "deepeval",
                "judge_model":  nesgen_judge.get_model_name(),
                "num_items":    len(eval_dataset),
                "threshold":    THRESHOLD,
            },
        )

        for metric_name, avg_score in avg_scores.items():
            trace.score(
                name=f"deepeval_{metric_name}",
                value=float(avg_score),
                comment=f"Average DeepEval {metric_name} score over {len(eval_dataset)} items",
            )

        lf.flush()
        print(f"Scores logged. Trace ID: {trace.id}")
    else:
        print("Langfuse authentication failed — skipping logging.")
else:
    print("Langfuse not configured — skipping. Set LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY to enable.")

## 14. Save Results to File

Export the evaluation results to a timestamped CSV for offline analysis.

In [ ]:
if "results_df" not in dir():
    raise RuntimeError(
        "results_df is not defined — run the batch evaluation cell (Section 9) first."
    )

from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join("..", "..", "metric_evaluation", "evaluation_results",
                           f"deepeval_results_{timestamp}.csv")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
results_df.to_csv(output_path, index=False)
print(f"Results saved to: {os.path.abspath(output_path)}")

## Summary

This notebook demonstrated:

1. **FIQA dataset** — loading the pre-built RAG evaluation split from Hugging Face (`explodinggradients/fiqa`), the same dataset used in the RAGAS cookbook
2. **NESGEN integration** — using NESGEN as the DeepEval judge LLM via `NESGENDeepEvalModel`
3. **7 RAG metrics** — Faithfulness, Answer Relevancy, Contextual Precision/Recall, Hallucination, Bias, Toxicity
4. **Single & batch evaluation** — per-item detailed results and aggregated averages
5. **High-level API** — using `DeepEvalEvaluator` for a simpler interface
6. **Native DeepEval evaluate()** — for CI/CD-ready evaluation with built-in reporting
7. **Langfuse logging** — optional integration for dashboards and monitoring

### Next Steps

- Replace simulated answers with your own RAG pipeline outputs
- Increase `DEMO_LIMIT` or remove `.select()` to run on the full FIQA dataset
- Explore additional DeepEval metrics: `GEval`, `SummarizationMetric`, `JsonCorrectnessMetric`, etc.
- Integrate this notebook into your CI/CD pipeline using `pytest` + DeepEval's pytest plugin